In [ ]:
import pandas as pd
import numpy as np

# 1. Nạp tập dữ liệu sạch từ Phase 2
df = pd.read_csv("clean_telemetry.csv")

# 2. Kiểm tra tổng quan kích thước và các trường thiếu dữ liệu
print(f"Kích thước tập dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột")
print("\nThống kê các trường khuyết thiếu (Top 5):")
print(df.isna().sum().sort_values(ascending=False).head(5))
df.head()

In [ ]:
# Loại bỏ cột 'notes' do chứa quá nhiều giá trị khuyết thiếu, không phục vụ mô hình
df = df.drop(columns=["notes"])
print(f"Kích thước sau khi loại bỏ cột không dùng: {df.shape}")

In [ ]:
# 1. Tái cấu trúc biến mục tiêu từ đa phân loại sang nhị phân (Target Engineering)
df['flight_target'] = df['flight_status'].apply(lambda x: "Completed" if x == "Completed" else "Non-completed")

# 2. Phân rã ma trận đặc trưng đầu vào X và biến mục tiêu y
# Loại bỏ các cột định danh hệ thống không mang tính quy luật vận hành
drop_cols = ["flight_status", "flight_target", "drone_id", "operator_id", "regulatory_approval_id", "flight_date"]
X = df.drop(columns=drop_cols)
y = df['flight_target']

print("Phân phối nhãn mục tiêu nhị phân phục vụ DSS:")
print(y.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

# Phân tách tập Train - Test theo tỷ lệ chuẩn 80/20 và giữ nguyên tỷ lệ phân phối nhãn
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Số lượng bản ghi huấn luyện (Train): {X_train.shape[0]}")
print(f"Số lượng bản ghi kiểm thử (Test): {X_test.shape[0]}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Phân loại tự động danh mục biến số thực và biến phân loại chữ
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

# Thiết lập đường ống xử lý tự động cho từng nhóm đặc trưng
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Gom hai nhánh xử lý vào bộ tiền xử lý trung tâm
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("✔ Khởi tạo Pipeline tiền xử lý dữ liệu tự động thành công!")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Thiết lập mô hình đối chiếu 1: Logistic Regression với class_weight balanced
log_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# Huấn luyện mô hình
log_model.fit(X_train, y_train)

# Dự báo và đánh giá định lượng
y_pred_log = log_model.predict(X_test)

print("=== KẾT QUẢ MÔ HÌNH LOGISTIC REGRESSION ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_log):.4f}")
print("\nMa trận nhầm lẫn (Confusion Matrix):")
print(confusion_matrix(y_test, y_pred_log))
print("\nBáo cáo phân loại chi tiết (Classification Report):")
print(classification_report(y_test, y_pred_log))

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

# Thiết lập mô hình đối chiếu 2: Random Forest Classifier với 200 cây quyết định
rf_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)), # Sử dụng clone để độc lập bộ nhớ với mô hình trước
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"))
])

# Huấn luyện mô hình
rf_model.fit(X_train, y_train)

# Dự báo và đánh giá định lượng
y_pred_rf = rf_model.predict(X_test)

print("=== KẾT QUẢ MÔ HÌNH RANDOM FOREST ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nMa trận nhầm lẫn (Confusion Matrix):")
print(confusion_matrix(y_test, y_pred_rf))
print("\nBáo cáo phân loại chi tiết (Classification Report):")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Tạo cấu trúc lưu trữ log siêu tham số hệ thống
hyperparams = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "max_iter": 1000,
        "class_weight": "balanced",
        "random_state": None,
        "n_estimators": None
    },
    {
        "model": "Random Forest",
        "max_iter": None,
        "class_weight": "balanced",
        "random_state": 42,
        "n_estimators": 200
    }
])

# Xuất tệp tin lưu vết phục vụ kiểm toán mô hình
hyperparams.to_csv("hyperparameters_log.csv", index=False)
print("[SUCCESS] Đã dọn dẹp hệ thống và kết xuất file 'hyperparameters_log.csv' thành công!")

In [ ]:
import joblib
joblib.dump(log_model, "output/logistic_model.pkl")
joblib.dump(rf_model, "output/random_forest_model.pkl")

In [ ]:
hyperparams.to_csv("output/hyperparameters.csv", index=False)

In [ ]:
import pandas as pd

results = pd.DataFrame([
    {"model": "Logistic Regression", "accuracy": 1.0},
    {"model": "Random Forest", "accuracy": 0.9905660377}
])

results.to_csv("output/model_results.csv", index=False)

In [ ]:
pred_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred_log
})
pred_df.to_csv("output/test_predictions.csv", index=False)

In [ ]:
joblib.dump(log_model, "output/logistic_pipeline.pkl")